# Phase 3 - SageMaker Managed Training

This notebook uploads the local training data and submits the churn model to SageMaker. It does not deploy an endpoint.

In [ ]:
from pathlib import Path
import os
import time
import boto3

os.environ['AWS_DEFAULT_REGION'] = 'eu-north-1'
REGION = 'eu-north-1'
ACCOUNT_ID = '717279701744'
BUCKET = f'sagemaker-ml-lab-{ACCOUNT_ID}-{REGION}'
REPO_DIR = Path('/home/sagemaker-user/shared/repos/c8xjjv9jiv7a8j/AWS-Lab_SageMaker/devs')

assert (REPO_DIR / 'src/training/train.py').exists()
assert (REPO_DIR / 'src/training/requirements.txt').exists()
print('Repository:', REPO_DIR)
print('Region:', REGION)
print('Bucket:', BUCKET)

In [ ]:
# Run repository tests before creating AWS resources.
%cd {REPO_DIR}
!pytest -q

In [ ]:
# Confirm the execution identity and required S3 input.
aws_session = boto3.Session(region_name=REGION)
sts = aws_session.client('sts')
s3 = aws_session.client('s3')
print(sts.get_caller_identity())
print(s3.head_bucket(Bucket=BUCKET))

training_data_uri = f's3://{BUCKET}/datasets/processed/'
training_object = f'{training_data_uri}train.csv'
print('Training data:', training_object)

In [ ]:
# Verify the uploaded input before submitting a paid training job.
s3.head_object(Bucket=BUCKET, Key='datasets/processed/train.csv')
print('S3 training input is available.')

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData

role = get_execution_role()
training_image = image_uris.retrieve(
    framework='sklearn',
    region=REGION,
    version='1.2-1',
    py_version='py3',
    instance_type='ml.m5.large',
    image_scope='training',
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    source_code=SourceCode(
        source_dir=str(REPO_DIR),
        entry_script='src/training/train.py',
        requirements='src/training/requirements.txt',
    ),
    compute=Compute(instance_type='ml.m5.large', instance_count=1),
)
print('Role:', role)
print('Image:', training_image)

In [ ]:
# This submits a SageMaker Training Job and incurs compute charges.
job = trainer.train(
    input_data_config=[
        InputData(channel_name='train', data_source=training_data_uri)
    ]
)
print(job)

In [ ]:
# Poll until the job completes or fails.
training_job_name = getattr(job, 'job_name', None) or getattr(job, 'training_job_name', None)
if not training_job_name:
    raise RuntimeError('Could not read the training job name from the SDK response')

sm = aws_session.client('sagemaker')
while True:
    details = sm.describe_training_job(TrainingJobName=training_job_name)
    status = details['TrainingJobStatus']
    print(status)
    if status in {'Completed', 'Failed', 'Stopped'}:
        break
    time.sleep(30)

if status != 'Completed':
    raise RuntimeError(details.get('FailureReason', f'Training job ended with {status}'))

model_artifact = details['ModelArtifacts']['S3ModelArtifacts']
print('Training job:', training_job_name)
print('Model artifact:', model_artifact)